# ACORec + AutoGluon

ACORec uses the current repository operator space (`ours`). Its proxy
receives the explicit outer train/validation split without splitting
the 80% search subset again, and query metafeatures are recomputed from
those permitted rows. The frozen recommendation is then evaluated by
AutoGluon on the untouched outer test split.

AutoGluon is limited to 300 seconds per evaluation in both modes.
Smoke mode only reduces the number of datasets. Runtime is recorded by
the runner and by this notebook.


In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO_URL = "https://github.com/MothMalone/SolutionRecommendation.git"
BRANCH = "experiment/aco-search-ablation"
REPO_DIR = Path("/kaggle/working/SolutionRecommendation")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-kaggle.txt")], check=True)
os.chdir(REPO_DIR)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
runner_source = (REPO_DIR / "scripts/run_recommend.py").read_text(encoding="utf-8")
evaluator_source = (REPO_DIR / "src/automl_aco/search/evaluation.py").read_text(encoding="utf-8")
required_markers = {
    "explicit proxy split": "_acorec_fixed_proxy_split" in runner_source and "_proxy_split_from_dataset" in evaluator_source,
    "split audit metadata": '"explicit_outer_split"' in runner_source,
    "leak-free query metafeatures": "computed_from_outer_train_validation" in runner_source,
}
missing = [name for name, present in required_markers.items() if not present]
if missing:
    raise RuntimeError(f"Remote checkout lacks required ACO protocol fixes: {missing}")
print("Protocol guard: explicit outer train/validation split; outer test excluded from ACO.")


In [ ]:
from __future__ import annotations
import json, os, shlex, shutil, subprocess, sys, time
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(REPO_DIR / "src"))
from automl_aco.eval_ids import EVAL_DATASETS
RUN_MODE = "smoke"       # change to final after smoke succeeds
NUM_SHARDS = 10; SHARD_INDEX = 0; WORKERS = 1
ACO_SEED = 42; SPLIT_SEED = 42; MAX_SAMPLES = 100_000
AG_TIME_LIMIT = 300; AG_PRESETS = "best_quality"
FINAL_N_ANTS, FINAL_N_ITERATIONS, FINAL_METRIC_EPOCHS = 10, 10, 100
if RUN_MODE not in {"smoke", "final"} or not 0 <= SHARD_INDEX < NUM_SHARDS: raise ValueError("Invalid mode/shard")
all_ids = [int(value) for value in EVAL_DATASETS.values()]
run_ids = all_ids[SHARD_INDEX::NUM_SHARDS]; run_ids = run_ids[:1] if RUN_MODE == "smoke" else run_ids
CACHE_DIR = Path("/kaggle/working/acorec_autogluon_data"); OUTPUT_DIR = Path(f"/kaggle/working/acorec_autogluon_{RUN_MODE}_shard_{SHARD_INDEX:02d}")
CACHE_DIR.mkdir(parents=True, exist_ok=True); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Mode={RUN_MODE}; shard={SHARD_INDEX}; ids={run_ids}; limit={AG_TIME_LIMIT}s")


In [ ]:
diffprep_names = {"abalone", "ada_prior", "avila", "connect-4", "eeg", "google", "house", "jungle_chess", "micro", "mozilla4", "obesity", "page-blocks", "pbcseq", "pol", "run_or_walk", "uscensus", "wall-robot-nav"}
expected_ids = {int(EVAL_DATASETS[name]) for name in diffprep_names}
command = [sys.executable, str(REPO_DIR / "scripts/export_diffprep_datasets.py"), "--out-dir", str(CACHE_DIR), "--download"]
input_root = Path("/kaggle/input")
attached_google = list(input_root.glob("**/google/data.csv")) if input_root.exists() else []
if attached_google: command[-1:] = ["--diffprep-root", str(input_root)]
subprocess.run(command, cwd=REPO_DIR, check=False)
present = {int(path.stem) for path in CACHE_DIR.glob("*.csv") if path.stem.isdigit()}
if expected_ids - present: raise RuntimeError(f"Missing DiffPrep snapshots: {sorted(expected_ids - present)}")
print(f"Frozen DiffPrep snapshots ready: {len(present)}")


In [ ]:
aco_command = [sys.executable, str(REPO_DIR / "scripts/run_recommend.py"), "--operator-space", "ours", "--performance-matrix", str(REPO_DIR / "data/openml/training_performance_matrix_autogluon.csv"), "--metafeatures", str(REPO_DIR / "data/openml/dataset_feats.csv"), "--pipeline-configs", str(REPO_DIR / "aco/pipeline_configs.json"), "--dataset-source", "openml", "--openml-backend", "gitlab", "--openml-local-folder", str(CACHE_DIR), "--dataset-ids", *[str(value) for value in run_ids], "--optimizer", "aco", "--seed", str(ACO_SEED), "--workers", str(WORKERS), "--output-dir", str(OUTPUT_DIR), "--skip-aco-plot", "--no-autogluon", "--recommend-on-train-val", "--recommend-split-seed", str(SPLIT_SEED), "--verbose"]
if RUN_MODE == "smoke": aco_command += ["--n-ants", "1", "--n-iterations", "1", "--no-train-metric-inline"]
else: aco_command += ["--n-ants", str(FINAL_N_ANTS), "--n-iterations", str(FINAL_N_ITERATIONS), "--train-metric-inline", "--metric-epochs", str(FINAL_METRIC_EPOCHS)]
print("ACORec command:\n", " ".join(shlex.quote(str(value)) for value in aco_command))


In [ ]:
env = os.environ.copy(); env.update({"PYTHONUNBUFFERED": "1", "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "NUMEXPR_NUM_THREADS": "1"})
started = time.perf_counter(); return_code = subprocess.run(aco_command, cwd=REPO_DIR, env=env, check=False).returncode
run_wall_clock_seconds = time.perf_counter() - started
if return_code != 0: raise RuntimeError(f"ACORec exited with code {return_code}")
print(f"ACORec wall-clock seconds: {run_wall_clock_seconds:.3f}")
for dataset_id in run_ids:
    dataset_dir = OUTPUT_DIR if len(run_ids) == 1 else OUTPUT_DIR / f"dataset_{dataset_id}"
    recommendation_path = dataset_dir / "recommendation.json"
    output_path = dataset_dir / "autogluon_evaluation.json"
    evaluation_command = [sys.executable, str(REPO_DIR / "scripts/evaluate_acorec_autogluon.py"), "--recommendation-json", str(recommendation_path), "--dataset-id", str(dataset_id), "--data-dir", str(CACHE_DIR), "--output-json", str(output_path), "--max-samples", str(MAX_SAMPLES), "--split-seed", str(SPLIT_SEED), "--time-limit", str(AG_TIME_LIMIT), "--presets", AG_PRESETS]
    print("Outer-test AutoGluon:", " ".join(shlex.quote(str(value)) for value in evaluation_command))
    subprocess.run(evaluation_command, cwd=REPO_DIR, env=env, check=True)


In [ ]:
def dataset_output_dir(dataset_id): return OUTPUT_DIR if len(run_ids) == 1 else OUTPUT_DIR / f"dataset_{dataset_id}"
rows = []
for dataset_id in run_ids:
    dataset_dir = dataset_output_dir(dataset_id)
    result = json.loads((dataset_dir / "recommendation.json").read_text(encoding="utf-8"))
    evaluation_path = dataset_dir / "autogluon_evaluation.json"
    evaluation = json.loads(evaluation_path.read_text(encoding="utf-8")) if evaluation_path.exists() else {}
    protocol = result.get("recommendation_protocol", {})
    rows.append({"dataset_id": int(dataset_id), "status": evaluation.get("status", "not_run"), "final_method": evaluation.get("method"), "score": evaluation.get("score"), "accuracy": evaluation.get("accuracy"), "proxy_split": protocol.get("proxy_split"), "proxy_train_rows": protocol.get("proxy_train_rows"), "proxy_validation_rows": protocol.get("proxy_validation_rows"), "outer_test_rows": protocol.get("outer_test_rows"), "query_metafeatures_source": protocol.get("query_metafeatures_source"), "aco_search_elapsed_seconds": result.get("elapsed_seconds"), "autogluon_total_seconds": evaluation.get("total_seconds"), "outer_evaluation_wall_clock_seconds": evaluation.get("acorec_and_evaluation_wall_clock_seconds"), "notebook_wall_clock_seconds": run_wall_clock_seconds, "autogluon_time_limit": AG_TIME_LIMIT, "error": evaluation.get("error", result.get("error"))})
summary = pd.DataFrame(rows); summary.to_csv(OUTPUT_DIR / "acorec_autogluon_summary.csv", index=False); display(summary)
archive = shutil.make_archive(str(Path("/kaggle/working") / OUTPUT_DIR.name), "gztar", root_dir=OUTPUT_DIR); print("Archive:", archive)
